# PA1 Tasks 2 + 3 - full re-run on Kaggle (nothing restored)

**Why.** Every Task 2 and Task 3 run is trained again from scratch under the handout protocol (seed 6304,
no gradient clipping, frozen BatchNorm statistics, AdamW 1e-4, at most 30 epochs, patience 5). Nothing from
an earlier session is loaded, so the results can be set against the originals to see whether anything
changes from run to run.

**What a re-run can and cannot tell you.** The seed is fixed, so this run differs from the original only
through numerical noise: different GPU (Colab vs Kaggle) and a few GPU kernels that are not bit-exact.
- A healthy run should land within about a point of the original.
- A collapse that happens again is systematic, not a fluke of one session.
- A result that **flips** (collapses once, trains the other time) sits on a knife-edge. That is "something
  else at play", and it belongs in the report.

A fixed-seed re-run cannot say whether seed 6304 itself was unlucky. **Part H1** checks that by repeating the
two MMD collapse runs with seed 6305. That is a labelled robustness check, never a comparison.

| Part | What happens | Runs trained |
|---|---|---|
| A | GPU check, fresh working copy, protocol, data, helpers | - |
| B | Task 2 training | source_only, dan, dann, cdan, dan λ 0.1 / 10, dann α_max 0.25 / 0.5 |
| C | Task 2 evaluation (the only use of Sketch labels in Task 2) | - |
| D | Task 3 training. **ERM = the source_only trained in Part B** | dan_dg, sam, dan_dg λ 0.1 / 10 |
| E | Task 3 source-side evaluation + collapse diagnosis (no Sketch) | - |
| F | Task 3 Sketch evaluation (the only Sketch access in Task 3) | - |
| G | Original vs re-run for every run, then save everything | - |
| H1 | Seed check with seed 6305 | dan λ 10 (Task 2), dan_dg λ 1 (Task 3) |
| H2 | Estimator switch: unbiased MMD, diagnostic. **Last cell** | diag_dan_dg_unbiased |

**Datasets to attach:** `pa1-repo`, `pa1-task2-b`, `pa1-task3` (**upload the v3 zip as a new version**),
`pacs-dataset`. You do **not** need `erm-checkpoint` or `task3-runs`: the Task 3 ERM is the Source-only model
re-trained in Part B, which is how the handout defines it.

**Timing.** Probably 3-5 hours on one GPU; every run prints its own training time. Kaggle allows 12 hours
per commit. Use **Save Version -> Save & Run All (Commit)**. Results are copied to `/kaggle/working/outputs/`
after every Part, so a failure late in the notebook does not lose the earlier Parts.

**Expected verdicts** (from the original runs): STABLE for source_only, dan, dan λ 0.1, sam, dan_dg λ 0.1;
COLLAPSED for dan λ 10, dann, dann α_max 0.5, dan_dg (λ 1), dan_dg λ 10; DEGRADED for cdan and dann α_max 0.25.
Part G flags every run whose verdict differs from the original.

## Part A - setup

In [ ]:
# A0. Hard GPU check -- before anything else. (An earlier session trained on the CPU without noticing.)
# Kaggle: Settings (right panel) -> Accelerator -> GPU P100 or GPU T4 x2, then Run All again.
import torch
if not torch.cuda.is_available():
    raise RuntimeError('NO GPU -- Settings -> Accelerator -> GPU, then Run All again.')
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

In [ ]:
# A1. Assemble a FRESH working copy of the repo from the three code datasets, then delete any results
# folders that came bundled with them. Nothing trained in an earlier session can be used below.
# (/kaggle/input is read-only; everything happens in /kaggle/working.)
import os, glob, shutil, json, math, subprocess, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

BASE = '/kaggle/input/datasets/zylah628'      # <- change if your datasets live elsewhere
REPO = '/kaggle/working/PA1'
OUT  = '/kaggle/working/outputs'
LAYERS = [f'{BASE}/pa1-repo',        # common/, shared/ (+ shared/splits/), task1/
          f'{BASE}/pa1-task2-b',     # task2/
          f'{BASE}/pa1-task3']       # task3/ + the updated shared/mmd.py, task2/methods/dan.py, task3/train.py

def root_of(d):
    if any(os.path.isdir(f'{d}/{x}') for x in ['task3', 'task2', 'common', 'shared']):
        return d
    subs = [p.rstrip('/') for p in glob.glob(f'{d}/*/') if os.path.isdir(p)]
    return subs[0] if len(subs) == 1 else d

os.makedirs(REPO, exist_ok=True); os.makedirs(OUT, exist_ok=True)
for layer in LAYERS:
    assert os.path.isdir(layer), f'Missing dataset: {layer}\nAttached under BASE: {os.listdir(BASE)}'
    shutil.copytree(root_of(layer), REPO, dirs_exist_ok=True, ignore=shutil.ignore_patterns('results'))
    print(f'  layered in: {os.path.basename(layer)}  (any results/ folder inside it was NOT copied)')
os.chdir(REPO)

# First time A1 runs in this session: make sure no results exist. If you re-run A1 later in the SAME
# session, the marker keeps the runs you have trained since then.
MARKER = f'{REPO}/.fresh_rerun_session'
if not os.path.exists(MARKER):
    for d in ['task2/results', 'task3/results']:
        if os.path.isdir(d):
            shutil.rmtree(d); print(f'  deleted old {d}/')
    open(MARKER, 'w').write(time.ctime())
    print('  fresh session: no earlier results exist in the working copy')
else:
    print('  A1 re-run inside the same session: keeping the runs trained since the session started')

PACS = os.path.dirname(glob.glob(f'{BASE}/pacs-dataset/**/sketch', recursive=True)[0])
T2_RUNS, T2_FINAL = 'task2/results/runs', 'task2/results/final'
T3_RUNS, T3_FINAL = 'task3/results/runs', 'task3/results/final'
T3_EVAL = f'--set data_root={PACS} erm_checkpoint={T2_RUNS}/source_only/best.pt'   # for Task 3's evaluation scripts

NEEDED = ['task2/train.py', 'task2/evaluate_final.py', 'task3/train.py', 'task3/evaluate_sources.py',
          'task3/evaluate_sketch.py', 'task3/diagnose_collapse.py', 'task3/configs/diag_dan_dg_unbiased.yaml',
          'shared/engine.py', 'shared/mmd.py', 'shared/splits/pacs_sources_seed6304.json',
          'shared/splits/pacs_sketch_target.json']
missing = [f for f in NEEDED if not os.path.exists(f)]
new_code = 'unbiased' in open('shared/mmd.py').read() and 'seed_check' in open('task3/train.py').read()
print(f'\n  repo    : {sorted(os.listdir(REPO))}')
print(f'  PACS    : {PACS}')
print(f'  v3 code : {"yes" if new_code else "NO -- upload the v3 zip as a new version of pa1-task3"}')
print(f'  MISSING : {missing if missing else "nothing -- ready"}')
assert not missing and new_code

In [ ]:
# A2. The protocol. Task 2 and Task 3 share task2/configs/base.yaml. Gradient clipping is set to 0.0: the
# handout specifies none, and the original main results (runs_clip0.0) used none.
os.system("sed -i 's/^grad_clip: .*/grad_clip: 0.0          # handout protocol: no gradient clipping/' task2/configs/base.yaml")
from common.io import load_config
t2, t3 = load_config('task2/configs/base.yaml'), load_config('task3/configs/base.yaml')
KEYS = ['seed', 'num_workers', 'per_domain_batch', 'target_batch', 'lr', 'weight_decay', 'grad_clip',
        'max_epochs', 'patience']
display(pd.DataFrame([{'setting': k, 'Task 2': t2.get(k), 'Task 3': t3.get(k)} for k in KEYS]))
print(f"Task 2 results -> {t2['results_dir']}")
print(f"Task 3 results -> {t3['results_dir']}")
print(f"Task 3 ERM     -> {t3['erm_checkpoint']}   (the Source-only model Part B is about to train)")
assert t2['grad_clip'] == 0.0 and t2['seed'] == 6304 and t3['erm_checkpoint'].startswith(T2_RUNS)

In [ ]:
# A3. Data check. The splits ship with the repo and must match this PACS copy image for image.
EXPECTED = {'photo': 1670, 'art_painting': 2048, 'cartoon': 2344, 'sketch': 3929}
for d, n in EXPECTED.items():
    got = len([p for p in glob.glob(f'{PACS}/{d}/*/*') if p.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'  {d:13s} {got:5d}  (expected {n})' + ('' if got == n else '   <-- MISMATCH, stop here'))
    assert got == n
src = json.load(open('shared/splits/pacs_sources_seed6304.json'))['domains']
tr = sum(len(s['train']) for s in src.values())
for d, s in src.items():
    print(f'  {d:13s} train={len(s["train"]):5d}  val={len(s["val"]):4d}')
print(f'\n  one epoch = {tr} // 24 = {tr // 24} updates; 30-epoch budget = {30 * (tr // 24)} updates')

In [ ]:
# A4. Helpers.
#   run(cmd)      runs a script and streams its output
#   train(...)    trains one run (skips it only if it was already trained IN THIS SESSION), then report()
#   report(...)   health summary + curves; returns a summary row
#   ORIGINAL      the numbers from the original sessions, for the run-to-run comparison in Part G
CHANCE_LOSS = math.log(7)   # 1.946: uniform guessing over 7 classes (a fresh head starts here)
PRIOR_LOSS = 1.913          # always predicting the source class frequencies: features carry no class info

def run(cmd):
    print('$', cmd, flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Command failed (exit code {p.returncode}): {cmd}')
    print(f'  [{(time.time() - t0) / 60:.1f} min]')

def verdict(s, hist):
    cls = [h['loss_cls'] for h in hist]; f1 = [h['val_mean_macro_f1'] for h in hist]
    gn = [h.get('grad_norm', float('nan')) for h in hist]
    if min(cls) > 1.6 or s['best_mean_macro_f1'] < 50:
        return 'COLLAPSED'
    if f1[-1] < s['best_mean_macro_f1'] - 10 or cls[-1] > max(0.5, 3 * min(cls)) or max(gn) > 1e3:
        return 'DEGRADED'
    return 'STABLE' if s['best_mean_macro_f1'] >= 90 else 'WEAK'

def report(runs_dir, run_name):
    '''What to look for:
       loss_cls   must end far below 1.95. Stuck at 1.91-1.95 = the head only learned class frequencies.
       loss_mmd   MMD penalty. The biased estimator cannot go below ~0.15 (24 vs 24, Task 2) or ~0.47
                  (8 vs 8, Task 3) even for identical domains, and its offset pulls features smaller.
       loss_dom   DANN/CDAN domain loss; 0.69 = discriminator guessing. Huge values = the runaway.
       loss_sam   SAM's loss at theta+eps; the gap above loss_cls should shrink.
       feat_norm  DAN-DG only: mean feature norm. Falling + loss_cls stuck = the collapse signature.'''
    d = f'{runs_dir}/{run_name}'
    s, hist = json.load(open(f'{d}/summary.json')), json.load(open(f'{d}/history.json'))
    ep = [h['epoch'] for h in hist]; f1 = [h['val_mean_macro_f1'] for h in hist]
    cls = [h['loss_cls'] for h in hist]; gn = [h.get('grad_norm', float('nan')) for h in hist]
    best_ep = ep[int(pd.Series(f1).idxmax())]
    minutes = sum(h.get('time_s', 0) for h in hist) / 60
    v = verdict(s, hist)
    print(f'\n=== {run_name} ===   ({minutes:.1f} min of training, seed {s["config"]["seed"]})')
    print(f'  epochs run              : {s["epochs_run"]} of {s["config"]["max_epochs"]}'
          f' ({"early stop" if s["epochs_run"] < s["config"]["max_epochs"] else "full budget"})')
    print(f'  best source-val macro-F1: {s["best_mean_macro_f1"]:.2f} (epoch {best_ep}), last epoch {f1[-1]:.2f}')
    print(f'  classification loss     : {cls[0]:.3f} -> {cls[-1]:.3f}  (min {min(cls):.3f}; 1.95 chance, 1.91 = class frequencies only)')
    for k, lab in [('loss_mmd', 'MMD penalty'), ('loss_dom', 'domain loss'), ('loss_sam', 'loss at theta+eps'),
                   ('feat_norm', 'feature norm |F(x)|')]:
        if k in hist[0]:
            print(f'  {lab:24s}: {hist[0][k]:.3f} -> {hist[-1][k]:.3f}')
    if 'disc_acc' in hist[0]:
        print(f'  discriminator accuracy  : {100*hist[0]["disc_acc"]:.1f}% -> {100*hist[-1]["disc_acc"]:.1f}%  (50% = confused)')
    print(f'  max gradient norm       : {max(gn):.3g}')
    orig = ORIGINAL.get(run_name)
    if orig and orig['f1'] is not None:
        flag = '' if orig['verdict'] == v else '   <-- VERDICT CHANGED'
        print(f'  ORIGINAL run            : F1 {orig["f1"]:.2f}, {orig["epochs"]} epochs, {orig["verdict"]}{flag}')
    print(f'  VERDICT: {v}')

    panels = [('loss_cls', 'classification loss', CHANCE_LOSS)]
    for k, t in [('loss_mmd', 'MMD penalty'), ('loss_dom', 'domain loss'), ('disc_acc', 'discriminator acc'),
                 ('loss_sam', 'loss at theta+eps'), ('feat_norm', 'feature norm')]:
        if k in hist[0]: panels.append((k, t, None))
    panels += [('val_mean_macro_f1', 'mean source-val macro-F1', None), ('grad_norm', 'gradient norm', None)]
    fig, axes = plt.subplots(1, len(panels), figsize=(3.1 * len(panels), 2.6))
    for ax, (k, title, line) in zip(axes, panels):
        y = [h.get(k, float('nan')) for h in hist]
        ax.plot(ep, y, marker='.')
        if k in ('grad_norm', 'loss_dom') and np.nanmax(y) > 100: ax.set_yscale('log')
        if line: ax.axhline(line, color='r', ls=':', lw=1)
        ax.set_title(f'{run_name}: {title}', fontsize=8); ax.set_xlabel('epoch'); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f'{OUT}/curves_{run_name}.png', dpi=130); plt.show(); plt.close(fig)
    return {'run': run_name, 'epochs': s['epochs_run'], 'best_epoch': best_ep,
            'best_src_val_f1': round(s['best_mean_macro_f1'], 2), 'last_src_val_f1': round(f1[-1], 2),
            'min_loss_cls': round(min(cls), 3), 'max_grad_norm': float(f'{max(gn):.3g}'),
            'train_minutes': round(minutes, 1), 'verdict': v}

summary_rows = {}
def train(task, config, run_name, extra=''):
    runs_dir = T2_RUNS if task == 2 else T3_RUNS
    if os.path.exists(f'{runs_dir}/{run_name}/summary.json'):
        print(f'{run_name}: already trained in THIS session -- skipping (nothing from earlier sessions exists here)')
    else:
        common = f'--set data_root={PACS}' + (f' erm_checkpoint={T2_RUNS}/source_only/best.pt' if task == 3 else '')
        run(f'python -m task{task}.train --config task{task}/configs/{config}.yaml {common} {extra}')
    summary_rows[run_name] = {'task': task, **report(runs_dir, run_name)}
    pd.DataFrame(summary_rows.values()).to_csv(f'{OUT}/training_summary.csv', index=False)

# Originals: Task 2 from the Colab runs_clip0.0 folder, Task 3 from the Kaggle CPU session.
ORIGINAL = {
    'source_only':      dict(task=2, epochs=20, f1=94.14, sketch=69.59, sep=99.86, verdict='STABLE'),
    'dan':              dict(task=2, epochs=15, f1=95.02, sketch=58.59, sep=82.28, verdict='STABLE'),
    'dann':             dict(task=2, epochs=23, f1=35.04, sketch=26.44, sep=98.08, verdict='COLLAPSED'),
    'cdan':             dict(task=2, epochs=6,  f1=88.75, sketch=58.49, sep=99.45, verdict='DEGRADED'),
    'dan_lambda0.1':    dict(task=2, epochs=13, f1=93.90, sketch=67.01, sep=97.25, verdict='STABLE'),
    'dan_lambda10':     dict(task=2, epochs=6,  f1=5.07,  sketch=4.07,  sep=68.27, verdict='COLLAPSED'),
    'dann_alpha0.25':   dict(task=2, epochs=6,  f1=88.75, sketch=52.30, sep=99.73, verdict='DEGRADED'),
    'dann_alpha0.5':    dict(task=2, epochs=6,  f1=44.74, sketch=15.47, sep=100.0, verdict='COLLAPSED'),
    'dan_dg':           dict(task=3, epochs=6,  f1=5.07,  sketch=None,  sep=None,  verdict='COLLAPSED'),
    'sam':              dict(task=3, epochs=13, f1=95.60, sketch=None,  sep=None,  verdict='STABLE'),
    'dan_dg_lambda0.1': dict(task=3, epochs=None, f1=None, sketch=None, sep=None, verdict='not finished'),
    'dan_dg_lambda10':  dict(task=3, epochs=None, f1=None, sketch=None, sep=None, verdict='never run'),
}
print('Helpers ready.')

## Part B - Task 2 training

Batches: 8 photo + 8 art + 8 cartoon (labelled) + 24 Sketch images **without labels** (the transductive UDA
protocol). Sketch labels are used only in Part C, after all eight runs are fixed.

In [ ]:
# B1. Source-only ERM. This model is ALSO the Task 3 ERM baseline (Part D compares against it).
# Expect STABLE, source-val F1 around 94.
train(2, 'source_only', 'source_only')

In [ ]:
# B2. The three alignment methods at their main settings.
#   dan   lambda_MMD = 1                        expect STABLE (~95)
#   dann  gradient reversal, alpha_max = 1      expect COLLAPSED: the reversed domain loss is unbounded
#   cdan  discriminator sees f (x) p            expect DEGRADED: learns, then blows up
for config, name in [('dan', 'dan'), ('dann', 'dann'), ('cdan', 'cdan')]:
    train(2, config, name)

In [ ]:
# B3. The declared controlled study: lambda_MMD in {0.1, 1, 10} (lambda = 1 is B2's dan).
# Expect lambda = 0.1 STABLE and lambda = 10 COLLAPSED (F1 5.07 = 'person' for every image).
for config, name in [('study_dan_lambda0.1', 'dan_lambda0.1'), ('study_dan_lambda10', 'dan_lambda10')]:
    train(2, config, name)

In [ ]:
# B4. The second study: DANN alpha_max in {0.25, 0.5, 1} (1 is B2's dann). In the original run both blew up
# within two epochs even though the reversal weight was tiny (~0.06): expect DEGRADED / COLLAPSED again.
for config, name in [('study_dann_alpha0.25', 'dann_alpha0.25'), ('study_dann_alpha0.5', 'dann_alpha0.5')]:
    train(2, config, name)

In [ ]:
# B5. All Task 2 runs, re-run vs original, on source information only.
def compare_rows(task):
    rows = []
    for r, o in ORIGINAL.items():
        if o['task'] != task or r not in summary_rows: continue
        n = summary_rows[r]
        rows.append({'run': r, 'orig_F1': o['f1'], 'rerun_F1': n['best_src_val_f1'],
                     'diff': None if o['f1'] is None else round(n['best_src_val_f1'] - o['f1'], 2),
                     'orig_epochs': o['epochs'], 'rerun_epochs': n['epochs'],
                     'orig_verdict': o['verdict'], 'rerun_verdict': n['verdict'],
                     'same_verdict': 'yes' if o['verdict'] == n['verdict'] else 'NO'})
    return pd.DataFrame(rows)
c2 = compare_rows(2); display(c2)
c2.to_csv(f'{OUT}/task2_training_rerun_vs_original.csv', index=False)
print('Healthy runs within about 1 F1 of the original and every verdict unchanged = reproducible.')
print('A changed verdict means that run sits on a knife-edge -- note it in the report.')

## Part C - Task 2 evaluation (Sketch labels are used only here)

Main table (Source-only, DAN, DANN, CDAN), the λ study, and the α_max study: source validation, Sketch
accuracy / macro-F1, domain separability (50% = chance), per-class changes and failure grids.

In [ ]:
# C1. Evaluate. Everything in Part B is fixed before this cell runs.
run('python -m task2.evaluate_final')
run('python -m task2.evaluate_final --study dan')
run('python -m task2.evaluate_final --study dann')

main2 = pd.read_csv(f'{T2_FINAL}/table_main.csv')
st_dan = pd.read_csv(f'{T2_FINAL}/table_study_dan.csv')
st_dann = pd.read_csv(f'{T2_FINAL}/table_study_dann.csv')
ev2 = pd.concat([main2, st_dan, st_dann]).drop_duplicates(subset='method').set_index('method')
display(main2.round(2))

print('\n=========== TASK 2: re-run vs original (Sketch accuracy and domain separability) ===========')
rows = []
for r, o in ORIGINAL.items():
    if o['task'] != 2 or r not in ev2.index: continue
    n = ev2.loc[r]
    rows.append({'run': r, 'orig_sketch': o['sketch'], 'rerun_sketch': round(n.target_acc, 2),
                 'diff': round(n.target_acc - o['sketch'], 2), 'orig_sep': o['sep'], 'rerun_sep': round(n.domain_sep, 2),
                 'rerun_src_F1': round(n.mean_src_f1, 2)})
c2e = pd.DataFrame(rows); display(c2e)
c2e.to_csv(f'{OUT}/task2_eval_rerun_vs_original.csv', index=False)
for f in glob.glob(f'{T2_FINAL}/*.csv') + glob.glob(f'{T2_FINAL}/*.json'):
    shutil.copy(f, f'{OUT}/task2_' + os.path.basename(f))
for f in glob.glob(f'{T2_FINAL}/figures/*.png'):
    shutil.copy(f, f'{OUT}/task2_' + os.path.basename(f))
print('\ncopied Task 2 tables and figures to', OUT, '(prefixed task2_)')

## Part D - Task 3 training (no Sketch from here until Part F)

ERM is **not** trained here: it is the Source-only model from Part B, as the handout requires.
`task3/train.py` refuses to start if any protocol setting differs from that checkpoint's.

In [ ]:
# D1. DAN-DG (lambda_DG = 1) and SAM (rho = 0.05): the main comparison.
#   dan_dg  expect COLLAPSED like the original (F1 5.07, loss_cls ~1.92, loss_mmd ~0.48)
#   sam     expect STABLE (~95-96), loss_sam gap above loss_cls shrinking
erm_sum = json.load(open(f'{T2_RUNS}/source_only/summary.json'))
print(f"ERM = this session's Source-only: best mean source-val macro-F1 {erm_sum['best_mean_macro_f1']:.2f}\n")
for config, name in [('dan_dg', 'dan_dg'), ('sam', 'sam')]:
    train(3, config, name)

In [ ]:
# D2. The Task 3 controlled study: lambda_DG in {0.1, 1, 10} (1 is D1's dan_dg).
# Expect lambda = 0.1 STABLE (the CPU run reached 92 F1 in 2 epochs) and lambda = 10 COLLAPSED.
for config, name in [('study_dan_dg_lambda0.1', 'dan_dg_lambda0.1'), ('study_dan_dg_lambda10', 'dan_dg_lambda10')]:
    train(3, config, name)
c3 = compare_rows(3); display(c3)
c3.to_csv(f'{OUT}/task3_training_rerun_vs_original.csv', index=False)

## Part E - Task 3 source-side evaluation and collapse diagnosis (still no Sketch)

In [ ]:
# E1. Per-domain / mean / worst source results, source-domain separability (33.3% = chance), sharpness proxy.
run(f'python -m task3.evaluate_sources {T3_EVAL}')
run(f'python -m task3.evaluate_sources --study dan_dg {T3_EVAL}')
ss = pd.read_csv(f'{T3_FINAL}/source_side.csv')
display(ss.round(3))
for _, r in ss.iterrows():
    print(f'  {r.method:10s} mean {r.mean_acc:6.2f}% | worst {r.worst_acc:6.2f}% '
          f'| source separability {r.src_domain_sep:6.2f}% | sharpness {r.sharpness:+.4f}')

In [ ]:
# E2. Why did DAN-DG collapse? Source validation only.
run(f'python -m task3.diagnose_collapse {T3_EVAL}')
cd = pd.read_csv(f'{T3_FINAL}/collapse_diagnostics.csv')
display(cd.round(3))
print('''How to read it
  top_pred_share ~100%      one class for everything (the head collapsed)
  class_probe high (>80%)   the FEATURES still separate the classes -> the head never learned to read them
  class_probe near 14.3%    the backbone itself erased class information
  feat_norm far below ERM   supports the shrinking explanation (biased-MMD pull)
  mmd_biased - floor        the part of the logged MMD that is a real domain difference''')
for f in glob.glob(f'{T3_FINAL}/*.csv') + glob.glob(f'{T3_FINAL}/*.json'):
    shutil.copy(f, f'{OUT}/task3_' + os.path.basename(f))

## Part F - Task 3 Sketch evaluation (the only Sketch access in Task 3)

In [ ]:
# F1. Main comparison, the lambda_DG study, per-class changes and the Task 2 vs Task 3 comparison (RQ4).
# The Task 2 side of RQ4 is this session's Part C, so both tasks come from the same re-run.
run(f'python -m task3.evaluate_sketch {T3_EVAL}')
run(f'python -m task3.evaluate_sketch --study dan_dg {T3_EVAL}')
main3 = pd.read_csv(f'{T3_FINAL}/table_main.csv')
display(main3.round(2))
erm_row = main3[main3.method == 'erm'].iloc[0]
for _, r in main3.iterrows():
    tag = 'baseline' if r.method == 'erm' else f'{r.sketch_acc - erm_row.sketch_acc:+.2f} pts vs ERM'
    print(f'  {r.method:10s} source mean {r.mean_f1:6.2f} F1 / worst {r.worst_f1:6.2f} | Sketch {r.sketch_acc:6.2f}% '
          f'| src-sep {r.src_domain_sep:6.2f}% | sharpness {r.sharpness:+.4f}   ({tag})')
st3 = pd.read_csv(f'{T3_FINAL}/table_study_dan_dg.csv')
display(st3.round(3))
print('\nThe main comparison keeps lambda_DG = 1 whatever the study shows.')
print('\n=== RQ4 ==='); print(open(f'{T3_FINAL}/task2_vs_task3_overall.json').read())
for f in glob.glob(f'{T3_FINAL}/*.csv') + glob.glob(f'{T3_FINAL}/*.json'):
    shutil.copy(f, f'{OUT}/task3_' + os.path.basename(f))
for f in glob.glob(f'{T3_FINAL}/figures/*.png'):
    shutil.copy(f, f'{OUT}/task3_' + os.path.basename(f))

## Part G - did anything change from run to run? Then save.

In [ ]:
# G1. Every run: original vs re-run.
allc = pd.concat([compare_rows(2).assign(task=2), compare_rows(3).assign(task=3)])
display(allc)
allc.to_csv(f'{OUT}/rerun_vs_original_all.csv', index=False)
changed = allc[allc.same_verdict == 'NO']
healthy = allc[(allc.rerun_verdict == 'STABLE') & allc['diff'].notna()]
print(f'\nVerdicts unchanged: {int((allc.same_verdict == "yes").sum())} of {len(allc)} runs'
      f'  (runs never finished originally count as changed: {list(allc[allc.orig_F1.isna()].run)})')
if len(healthy):
    print(f'Healthy runs: largest source-F1 difference from the original = {healthy["diff"].abs().max():.2f} points')
real = changed[changed.orig_F1.notna()]
if len(real):
    print('\nRUNS WHOSE VERDICT CHANGED (knife-edge -- report them):')
    for _, r in real.iterrows():
        print(f'  {r.run:18s} {r.orig_verdict} (F1 {r.orig_F1}) -> {r.rerun_verdict} (F1 {r.rerun_F1})')
else:
    print('\nNo finished run changed verdict: the collapses are reproducible, not session flukes.')

In [ ]:
# G2. Save. The Task 2 folders are exported as runs_rerun / final_rerun so they cannot overwrite the original
# runs_clip0.0 / final_clip0.0 in your GitHub repo. Task 3 keeps its default folder names (its scripts use them).
def export(tag):
    stage = f'/kaggle/working/export_{tag}'
    if os.path.isdir(stage): shutil.rmtree(stage)
    for z in ['rerun_light.zip', 'rerun_full.zip']:          # rebuilt from scratch each time
        if os.path.exists(f'{OUT}/{z}'): os.remove(f'{OUT}/{z}')
    for src_dir, dst in [(T2_RUNS, 'task2/results/runs_rerun'), (T2_FINAL, 'task2/results/final_rerun'),
                         (T3_RUNS, 'task3/results/runs'), (T3_FINAL, 'task3/results/final')]:
        if os.path.isdir(src_dir):
            shutil.copytree(src_dir, f'{stage}/{dst}')
    os.system(f'cd {stage} && zip -qr {OUT}/rerun_light.zip . -x "*.pt"')     # tables, histories, figures
    os.system(f'cd {stage} && zip -qr {OUT}/rerun_full.zip .')                # + checkpoints (large)
    shutil.rmtree(stage)
export('main')
for f in sorted(os.listdir(OUT)):
    print(f'{os.path.getsize(f"{OUT}/{f}")/1e6:8.2f} MB  outputs/{f}')
print('\nDownload rerun_light.zip + the CSVs; rerun_full.zip only if you want the checkpoints.')

## Part H1 - seed check (robustness, not a comparison)

The two MMD collapses are repeated with **seed 6305**. That changes the head's initialisation, the
sampling order and the augmentation; the splits stay fixed. If they collapse again, the collapse does not
depend on seed 6304. Neither run is ever evaluated on Sketch or used in any table of the main comparison.

In [ ]:
# H1. Seed check.
SEED_CHECK = True               # set to False to skip
CHECK_SEED = 6305
SEED_RUNS = [(2, 'study_dan_lambda10', 'dan_lambda10'),    # Task 2 MMD collapse
             (3, 'dan_dg', 'dan_dg')]                       # Task 3 MMD collapse (lambda_DG = 1)
if not SEED_CHECK:
    print('Skipped (SEED_CHECK = False).')
else:
    rows = []
    for task, config, base in SEED_RUNS:
        name = f'{base}_seed{CHECK_SEED}'
        extra = f'seed={CHECK_SEED} run_name={name}' + (' seed_check=true' if task == 3 else '')
        train(task, config, name, extra)
        a, b = summary_rows[base], summary_rows[name]
        rows.append({'run': base, 'seed 6304 F1': a['best_src_val_f1'], 'seed 6304 verdict': a['verdict'],
                     f'seed {CHECK_SEED} F1': b['best_src_val_f1'], f'seed {CHECK_SEED} verdict': b['verdict'],
                     'same': 'yes' if a['verdict'] == b['verdict'] else 'NO'})
    sc = pd.DataFrame(rows); display(sc)
    sc.to_csv(f'{OUT}/seed_check.csv', index=False)
    print('same = yes: the collapse does not depend on the seed.  same = NO: it does -- report that.')
    export('seed')

## Part H2 - the estimator switch (diagnostic, last on purpose)

DAN-DG again at **λ_DG = 1**, with the same kernels, bandwidth, batches and protocol. Only the MMD estimator
changes: **unbiased**, so no self-comparisons, no ~0.47 floor and no pull that keeps shrinking the features.
It runs after the Task 3 Sketch evaluation, so the main comparison is already locked with the biased
estimator. Report this as a diagnostic only. It is never evaluated on Sketch.

| If the unbiased run... | it means |
|---|---|
| trains (source F1 around 90, `loss_cls` well below 1.9) | the biased estimator's small-sample pull is the most likely cause of the collapse |
| collapses the same way (F1 about 5, `loss_cls` about 1.92) | the estimator is not the cause; the MMD gradient at 8 vs 8 is too strong or noisy either way |

In [ ]:
# H2. LAST CELL -- the estimator switch, as a labelled DIAGNOSTIC. Everything above is already saved.
RUN_UNBIASED_DIAG = True        # set to False to skip
DIAG = 'diag_dan_dg_unbiased'
if not RUN_UNBIASED_DIAG:
    print('Skipped (RUN_UNBIASED_DIAG = False).')
else:
    train(3, DIAG, DIAG)
    run(f'python -m task3.diagnose_collapse {T3_EVAL}')   # the table now includes the diagnostic run
    cd = pd.read_csv(f'{T3_FINAL}/collapse_diagnostics.csv').set_index('run')
    display(cd.round(3))

    hb, hu = [json.load(open(f'{T3_RUNS}/{r}/history.json')) for r in ('dan_dg', DIAG)]
    sb, su = [json.load(open(f'{T3_RUNS}/{r}/summary.json')) for r in ('dan_dg', DIAG)]
    def line(label, a, b, fmt='{:.3f}'):
        print(f'  {label:48s}{fmt.format(a):>14s}{fmt.format(b):>16s}')
    print('\n============ lambda_DG = 1: biased (main) vs unbiased (diagnostic) ============')
    print(f'  {"":48s}{"biased/main":>14s}{"unbiased/diag":>16s}')
    line('best mean source-val macro-F1', sb['best_mean_macro_f1'], su['best_mean_macro_f1'], '{:.2f}')
    line('epochs run', sb['epochs_run'], su['epochs_run'], '{:d}')
    line('lowest loss_cls (1.913 = no class info)', min(h['loss_cls'] for h in hb), min(h['loss_cls'] for h in hu))
    line('last loss_mmd (biased floor ~0.47; unbiased ~0)', hb[-1]['loss_mmd'], hu[-1]['loss_mmd'])
    line('first -> last feature norm (biased run)', hb[0]['feat_norm'], hb[-1]['feat_norm'])
    line('first -> last feature norm (unbiased run)', hu[0]['feat_norm'], hu[-1]['feat_norm'])
    line('max gradient norm', max(h['grad_norm'] for h in hb), max(h['grad_norm'] for h in hu), '{:.3g}')
    if {'dan_dg', DIAG} <= set(cd.index):
        line('class probe on frozen features (%, 14.3 = chance)', cd.loc['dan_dg', 'class_probe'], cd.loc[DIAG, 'class_probe'], '{:.1f}')
        line('share of images given the top class (%)', cd.loc['dan_dg', 'top_pred_share'], cd.loc[DIAG, 'top_pred_share'], '{:.1f}')
        line('source-domain separability (%, 33.3 = chance)', cd.loc['dan_dg', 'domain_sep'], cd.loc[DIAG, 'domain_sep'], '{:.1f}')

    f1u = su['best_mean_macro_f1']
    print('\nWHAT IT MEANS')
    if f1u >= 80:
        print("  The unbiased run TRAINED: the biased estimator's small-sample pull is the most likely cause of the")
        print('  lambda = 1 collapse. The main DAN-DG row stays the biased run; report this as a diagnostic.')
    elif f1u < 50:
        print('  The unbiased run ALSO COLLAPSED: the estimator is not the cause. The MMD gradient from 8-vs-8')
        print('  estimates overwhelms classification either way. Keep the biased results and report this.')
    else:
        print("  IN BETWEEN: it learned something but not well. Send this cell's output to Claude.")
    shutil.copy(f'{T3_FINAL}/collapse_diagnostics.csv', f'{OUT}/task3_collapse_diagnostics.csv')
    export('final')
    print('\nre-saved: the zips now include every run, the seed check and this diagnostic.')